In [ ]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID = 0 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU 0:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
from datetime import datetime

In [ ]:
sys.path.insert(0, "/workspace/app")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from coding.Paper_CBMS import Run_RNN_dropout as rf
from sklearn.metrics import mean_absolute_error, mean_squared_error, recall_score, precision_score, f1_score, average_precision_score, precision_recall_fscore_support, roc_auc_score

In [ ]:
def build_risk_curve_df_from_result(res):
    return pd.DataFrame({
        "Patient_ID": res["Patient_ID"],
        "Session":    res["Session"],
        "p_next":     res["y_prob"],
        "y_h2":       res.get("y_true", None)
    }).sort_values("Session")

In [ ]:
def early_warning_metrics_for_patient(risk_curve_df, tau, dropout_session):
    crossed = risk_curve_df[risk_curve_df["p_next"] >= tau]
    warn_t = int(crossed["Session"].min()) if len(crossed) else None
    if dropout_session is None or (isinstance(dropout_session, float) and np.isnan(dropout_session)):
        return {"warn_t": warn_t, "lead_sessions": None, "detected": None}
    event_prev = int(dropout_session) - 1  
    if warn_t is None:
        return {"warn_t": None, "lead_sessions": None, "detected": 0}
    lead = event_prev - warn_t
    detected = 1 if lead >= 0 else 0
    return {"warn_t": warn_t, "lead_sessions": lead, "detected": detected}

In [ ]:
def summarize_early_warning(all_patient_metrics):
    dropouts = [m for m in all_patient_metrics if m.get("is_dropout") == 1]
    if not dropouts:
        return {}

    any_alert = [1 if m.get("warn_t") is not None else 0 for m in dropouts]
    any_alert_rate = float(np.mean(any_alert)) if any_alert else 0.0

    def lead_val(m):
        ls = m.get("lead_sessions", None)
        if ls is None: 
            return None
        if isinstance(ls, float) and np.isnan(ls):
            return None
        return int(ls)

    leads_all = [lead_val(m) for m in dropouts]

    def rate_ge(k):
        vals = []
        for l in leads_all:
            vals.append(1 if (l is not None and l >= k) else 0)
        return float(np.mean(vals)) if vals else 0.0

    timely_detection_rate = rate_ge(0)
    rate_lead_ge_1 = rate_ge(1)
    rate_lead_ge_2 = rate_ge(2)

    leads_timely = [l for l in leads_all if (l is not None and l >= 0)]
    mean_lead_timely = float(np.mean(leads_timely)) if leads_timely else None

    return {
        "any_alert_rate": any_alert_rate,
        "timely_detection_rate": timely_detection_rate,  # = rate_lead_ge_0
        "rate_lead_ge_0": timely_detection_rate,
        "rate_lead_ge_1": rate_lead_ge_1,
        "rate_lead_ge_2": rate_lead_ge_2,
        "mean_lead_sessions_detected": mean_lead_timely
    }

In [ ]:
def select_three_examples_informative(results):
    """
    Returns:
      best_early: dropout with max lead (detected==1)
      worst_or_missed: dropout with detected==0 if exists, else detected with min lead
      hard_completer: completer with highest max p_next (likely FP-ish)
    """
    dropouts = [r for r in results if r and r.get("dropout_session") is not None]
    completers = [r for r in results if r and r.get("dropout_session") is None]

    # compute EW stats for dropouts
    scored_dropouts = []
    for r in dropouts:
        curve = build_risk_curve_df_from_result(r)
        tau = r.get("tau", 0.6)
        m = early_warning_metrics_for_patient(curve, tau, r.get("dropout_session"))
        scored_dropouts.append((r, m))

    # best early = detected with max lead
    detected = [(r, m) for (r, m) in scored_dropouts if m["detected"] == 1 and m["lead_sessions"] is not None]
    if detected:
        best_early = max(detected, key=lambda x: x[1]["lead_sessions"])[0]
    else:
        # fallback: just pick any dropout
        best_early = scored_dropouts[0][0] if scored_dropouts else None

    # worst = missed if exists; else min lead among detected
    missed = [(r, m) for (r, m) in scored_dropouts if m["detected"] == 0]
    if missed:
        worst_or_missed = missed[0][0]  # any missed; could also pick one with largest dropout_session
    else:
        if detected:
            worst_or_missed = min(detected, key=lambda x: x[1]["lead_sessions"])[0]
        else:
            worst_or_missed = scored_dropouts[-1][0] if scored_dropouts else None

    # hard completer = highest max risk (most likely FP / alarming)
    if completers:
        def completer_score(r):
            y_prob = np.asarray(r.get("y_prob", []), dtype=float)
            if y_prob.size == 0:
                return -1.0
            return float(np.nanmax(y_prob))
        hard_completer = max(completers, key=completer_score)
    else:
        hard_completer = None

    return best_early, worst_or_missed, hard_completer

In [ ]:
def plot_three_risk_curves_informative(results):
    best, worst, comp = select_three_examples_informative(results)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
    for ax, res, title in zip(
        axes,
        [best, worst, comp],
        ["Best early-warning dropout", "Missed/late-warning dropout", "Hard completer (FP-like)"]
    ):
        if res is None:
            ax.set_title(title + "\n(None)")
            ax.set_xlim(1, 8)
            ax.set_ylim(0, 1)
            continue
        curve = build_risk_curve_df_from_result(res)
        ax.plot(curve["Session"], curve["p_next"], marker="o")
        tau = res.get("tau", None)
        if tau is not None:
            ax.axhline(y=tau, linestyle="--")
        ds = res.get("dropout_session", None)
        if ds is not None and not (isinstance(ds, float) and np.isnan(ds)):
            ax.axvline(x=int(ds), linestyle=":")
        ax.set_title(f"{title}\nPatient {res['Patient_ID']}")
        ax.set_xlabel("Session")
        ax.set_ylim(0, 1)
        ax.set_xlim(1, 8)
        ax.set_xticks(range(1, 9))
    axes[0].set_ylabel("p_next")
    plt.tight_layout()
    plt.show()

Datasets

* Experiment 3: Ablation study: best calibrated vocal + Process engagement (scheduling) data + 
- elapsed_time_so_far (time between S1 and session t)
- mean_gap_so_far (average of the gaps between sessions observed up to t)
- std_gap_so_far

Increased recall rate versus false alarms when adding engagement - scheduling irregularities amplify the signal.

In [ ]:
df = pd.read_excel("/workspace/app/planilhas/CBMS/df_all_Horizon2.xlsx")

In [ ]:
df = df[df["Embedding_npy"].notna() & df["Embedding_npy"].apply(lambda x: isinstance(x, str))].copy()
df = df[df["Patient_ID"] != 2].copy()

In [ ]:
cols_remove = ['Age', "Patient_Gender", 'Education', 'Y_Zscore_TO_TOT_Alexithymia', 'Questionnary5_HQ_25_T0_soc_Score', 
              'Questionnary5_HQ_25_T0_iso_Score', 'Questionnary5_HQ_25_T0_sup_Score', 'Mean_Gap_Days', 'Std_Gap_Days', 
              'Y_Depression_T0', 'Embedding_npy_list']
df = df.drop(columns=cols_remove, errors="ignore")

In [ ]:
mfcc_mean = [f"MFCC{s}_mean" for s in range(1, 14)]

Engagement Data

In [ ]:
df_eng = pd.read_excel('/workspace/app/planilhas/CBMS/df_eng.xlsx')
df_merge = pd.merge(df, df_eng, on=["Patient_ID", "Session"])

In [ ]:
df_final = df_merge.copy()
tabular = mfcc_mean + ["F0_mean", "F0_std", 'elapsed_time_so_far', 'mean_gap_so_far', 'std_gap_so_far'] 
cal_selected = 'f2'
warn_selected = '30perc'
BS = 40
epochs = 10

GRU

In [ ]:
summary_results = []
all_results = []
target_column = "y_h2" 
unique_patients = df['Patient_ID'].unique() 
results_gru = [rf.class_lopo_RNN_dropout(p, df_final, 'Patient_ID', target_column, "GRU", tab_cols = tabular, BS=BS, epochs=epochs, calibrate_tau=cal_selected, warn_rule=warn_selected)
    for p in unique_patients]
all_results.extend(results_gru)
df_current = pd.DataFrame(results_gru)
df_current["Accuracy"] = np.where(
    df_current["y_patient_true"] == 1,
    (df_current["y_patient_pred"] == 1).astype(float),
    (df_current["y_patient_pred"] == 0).astype(float))
df_current["BalAcc"] = (df_current["REC"].fillna(0) + df_current["SPEC"].fillna(0)) / 2
summary_results.append({
        "Accuracy_Macro_Mean": df_current["Accuracy"].mean(),
        "Accuracy_Macro_Std":  df_current["Accuracy"].std(),
        "BalAcc_Mean":          df_current["BalAcc"].mean(),
        "BalAcc_Std":           df_current["BalAcc"].std(),
        "Recall_Pos_Mean":     df_current.loc[df_current["y_patient_true"] == 1, "REC"].mean(),
        "Spec_Pos_Mean":       df_current["SPEC"].mean(), 
        "Precision_Pos_Mean":  df_current["PREC"].mean(),  
        "F1_Pos_Mean":         df_current["F1"].mean()})
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)

In [ ]:
df_results.to_excel('/workspace/app/planilhas/CBMS/Exp3_GRU.xlsx', index=False)
df_summary.to_excel('/workspace/app/planilhas/CBMS/Exp3_GRU_SUMMARY.xlsx', index=False)

In [ ]:
print("TP:", ((df_results.y_patient_true==1)&(df_results.y_patient_pred==1)).sum())
print("FN:", ((df_results.y_patient_true==1)&(df_results.y_patient_pred==0)).sum())
print("FP:", ((df_results.y_patient_true==0)&(df_results.y_patient_pred==1)).sum())
print("TN:", ((df_results.y_patient_true==0)&(df_results.y_patient_pred==0)).sum())

In [ ]:
all_patient_metrics = []
for res in results_gru:
    if res is None: 
        continue
    curve = build_risk_curve_df_from_result(res)
    m = early_warning_metrics_for_patient(curve, res["tau"], res.get("dropout_session", None))
    ds = res.get("dropout_session", None)
    is_dropout = 0 if (ds is None or (isinstance(ds, float) and np.isnan(ds))) else 1
    all_patient_metrics.append({
        "Patient_ID": res["Patient_ID"],
        "is_dropout": is_dropout,
        **m
    })

In [ ]:
summary = summarize_early_warning(all_patient_metrics)
print(summary)

In [ ]:
plot_three_risk_curves_informative(results_gru)